In [1]:
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()  # repository root
(ROOT / "results" / "figures").mkdir(parents=True, exist_ok=True)
# ============================================================
# STEP 1: LOAD AND INSPECT DATASET NO. 1
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder

RANDOM_STATE = 42

# Load dataset
data = pd.read_csv(ROOT / "data" / "Dataset No.1.csv")

print("Dataset shape:", data.shape)
print("\nColumns:")
print(data.columns.tolist())

print("\nFirst 5 rows:")
display(data.head())

print("\nMissing values:")
missing = data.isnull().sum()
print(missing[missing > 0])

print("\nOriginal Behavior distribution:")
print(data["Behavior"].value_counts())

print("\nOriginal Behavior proportions:")
print((data["Behavior"].value_counts(normalize=True) * 100).round(2))

print("\nNumber of exact duplicate rows:")
print(data.duplicated().sum())

print("\nUnique ASTM materials:")
print(data["ASTM Material"].nunique())

Dataset shape: (4019, 31)

Columns:
['ASTM Material', 'C', 'Mn', 'Si', 'P', 'S', 'Cr', 'Mo', 'Ni', 'N', 'Ti', 'Nb ', 'Al', 'Fe balance (%)', 'Formic acid (HCOOH)', 'Ammonium chloride(NH4Cl)', 'Acetic acid(CH3COOH)', 'Potasium Hydroxide (KOH)', 'Lactic acid ', 'oxalic acid (COOH)2.2H2O', 'Phosphric acid (H3PO4)', 'Sulfuric acid (H2SO4)', 'Nitric acid (HNO3)', 'Hydrocloric acid (HCl)', 'Citric acid(HOC(CH2COOH)2COOH.H2O', 'KHSO4', 'KNO3', 'MgCl2.6H2O', 'Temperature', 'Behavior', 'Pitting_label']

First 5 rows:


,ASTM Material,C,Mn,Si,P,S,Cr,Mo,Ni,N,...,Sulfuric acid (H2SO4),Nitric acid (HNO3),Hydrocloric acid (HCl),Citric acid(HOC(CH2COOH)2COOH.H2O,KHSO4,KNO3,MgCl2.6H2O,Temperature,Behavior,Pitting_label
0,403,0.15,1.00,0.5,0.040,0.03,11.5,0.0,0.60,0.0,...,0.0,0,0.0,0,0,0,0,20.0,Poor,0
1,405,0.08,1.00,1.0,0.040,0.03,11.5,0.0,0.60,0.0,...,0.0,0,0.0,0,0,0,0,20.0,Poor,0
2,S41050,0.04,1.00,1.0,0.045,0.03,10.5,0.0,0.60,0.1,...,0.0,0,0.0,0,0,0,0,20.0,Poor,0
3,416,0.15,1.25,1.0,0.060,0.35,12.0,0.0,0.00,0.0,...,0.0,0,0.0,0,0,0,0,20.0,Poor,0
4,410,0.15,1.00,1.0,0.040,0.03,11.5,0.0,0.75,0.0,...,0.0,0,0.0,0,0,0,0,20.0,Poor,0



Missing values:
Series([], dtype: int64)

Original Behavior distribution:
Behavior
Resistant    1311
Severe       1233
Poor          753
Good          722
Name: count, dtype: int64

Original Behavior proportions:
Behavior
Resistant    32.62
Severe       30.68
Poor         18.74
Good         17.96
Name: proportion, dtype: float64

Number of exact duplicate rows:
0

Unique ASTM materials:
34


In [2]:
# ============================================================
# STEP 2: PREPARE FEATURES AND CREATE LOCKED TEST SET
# ============================================================

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

RANDOM_STATE = 42
TEST_SIZE = 0.20

# ------------------------------------------------------------
# 1. Make a clean working copy
# ------------------------------------------------------------
df = data.copy()

# Clean accidental spaces in column names
df.columns = df.columns.str.strip()

# ------------------------------------------------------------
# 2. Encode target
# ------------------------------------------------------------
label_encoder = LabelEncoder()

y = pd.Series(
    label_encoder.fit_transform(df["Behavior"]),
    index=df.index,
    name="Behavior"
)

print("Class encoding:")
for i, label in enumerate(label_encoder.classes_):
    print(f"  {i} = {label}")

# ------------------------------------------------------------
# 3. Prepare features
# ------------------------------------------------------------

# Keep ASTM Material separately for later group-based validation
groups = df["ASTM Material"].copy()

# Remove identifier, target and alternative target
X = df.drop(
    columns=["ASTM Material", "Behavior", "Pitting_label"]
).copy()

print("\nNumber of candidate features:", X.shape[1])

# Make sure every predictor is numeric
non_numeric = X.select_dtypes(exclude=np.number).columns.tolist()

print("Non-numeric feature columns:", non_numeric)

if non_numeric:
    raise ValueError(
        f"Unexpected non-numeric features found: {non_numeric}"
    )

# ------------------------------------------------------------
# 4. Stratified 80:20 train-test split
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

# ------------------------------------------------------------
# 5. Sanity checks
# ------------------------------------------------------------

print("\nDataset sizes:")
print(f"Full dataset : {len(X)}")
print(f"Training set : {len(X_train)}")
print(f"Test set     : {len(X_test)}")

print("\nTraining class distribution:")
train_distribution = pd.DataFrame({
    "Count": y_train.value_counts().sort_index(),
    "Percentage": (
        y_train.value_counts(normalize=True)
        .sort_index() * 100
    ).round(2)
})
train_distribution.index = [
    label_encoder.classes_[i]
    for i in train_distribution.index
]
display(train_distribution)

print("\nTest class distribution:")
test_distribution = pd.DataFrame({
    "Count": y_test.value_counts().sort_index(),
    "Percentage": (
        y_test.value_counts(normalize=True)
        .sort_index() * 100
    ).round(2)
})
test_distribution.index = [
    label_encoder.classes_[i]
    for i in test_distribution.index
]
display(test_distribution)

# ------------------------------------------------------------
# 6. Verify that train and test rows do not overlap
# ------------------------------------------------------------

overlap = X_train.index.intersection(X_test.index)

print("\nTrain-test row overlap:", len(overlap))

assert len(overlap) == 0
assert len(X_train) + len(X_test) == len(X)

print("\nLocked train-test split created successfully.")

Class encoding:
  0 = Good
  1 = Poor
  2 = Resistant
  3 = Severe

Number of candidate features: 28
Non-numeric feature columns: []

Dataset sizes:
Full dataset : 4019
Training set : 3215
Test set     : 804

Training class distribution:


,Count,Percentage
Good,578,17.98
Poor,602,18.72
Resistant,1049,32.63
Severe,986,30.67



Test class distribution:


,Count,Percentage
Good,144,17.91
Poor,151,18.78
Resistant,262,32.59
Severe,247,30.72



Train-test row overlap: 0

Locked train-test split created successfully.


In [3]:
# ============================================================
# STEP 3: CLEAN BASELINE MODELS
# Dataset No. 1
# ============================================================

import time
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from xgboost import XGBClassifier


# ------------------------------------------------------------
# 1. Define baseline models
# ------------------------------------------------------------

baseline_models = {

    "Decision Tree": DecisionTreeClassifier(
        random_state=RANDOM_STATE
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),

    "Bagging": BaggingClassifier(
        n_estimators=100,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),

    "XGBoost": XGBClassifier(
        n_estimators=100,
        random_state=RANDOM_STATE,
        eval_metric="mlogloss",
        n_jobs=-1
    )
}


# ------------------------------------------------------------
# 2. Train and evaluate each baseline model
# ------------------------------------------------------------

baseline_results = []
fitted_baseline_models = {}

for name, model in baseline_models.items():

    print("=" * 70)
    print(f"Training: {name}")
    print("=" * 70)

    model = clone(model)

    start = time.time()

    # Train ONLY on training data
    model.fit(X_train, y_train)

    runtime = time.time() - start

    # Predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # Predicted probabilities for multiclass ROC-AUC
    y_test_proba = model.predict_proba(X_test)

    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc = accuracy_score(y_test, y_test_pred)

    gap = train_acc - test_acc

    macro_precision = precision_score(
        y_test,
        y_test_pred,
        average="macro",
        zero_division=0
    )

    macro_recall = recall_score(
        y_test,
        y_test_pred,
        average="macro",
        zero_division=0
    )

    macro_f1 = f1_score(
        y_test,
        y_test_pred,
        average="macro",
        zero_division=0
    )

    macro_auc = roc_auc_score(
        y_test,
        y_test_proba,
        multi_class="ovr",
        average="macro"
    )

    # Save model
    fitted_baseline_models[name] = model

    # Save results
    baseline_results.append({
        "Model": name,
        "Train Accuracy": train_acc,
        "Test Accuracy": test_acc,
        "Gap": gap,
        "Macro Precision": macro_precision,
        "Macro Recall": macro_recall,
        "Macro F1": macro_f1,
        "Macro ROC-AUC": macro_auc,
        "Runtime (s)": runtime
    })

    print(f"Train Accuracy : {train_acc:.4f}")
    print(f"Test Accuracy  : {test_acc:.4f}")
    print(f"Train-Test Gap : {gap:.4f}")
    print(f"Macro Precision: {macro_precision:.4f}")
    print(f"Macro Recall   : {macro_recall:.4f}")
    print(f"Macro F1       : {macro_f1:.4f}")
    print(f"Macro ROC-AUC  : {macro_auc:.4f}")
    print(f"Runtime        : {runtime:.2f} s")
    print()


# ------------------------------------------------------------
# 3. Create comparison table
# ------------------------------------------------------------

baseline_results_df = pd.DataFrame(baseline_results)

baseline_results_df = baseline_results_df.sort_values(
    by="Test Accuracy",
    ascending=False
).reset_index(drop=True)


# Percentage formatting for easier reading
display_df = baseline_results_df.copy()

percentage_columns = [
    "Train Accuracy",
    "Test Accuracy",
    "Gap",
    "Macro Precision",
    "Macro Recall",
    "Macro F1",
    "Macro ROC-AUC"
]

for col in percentage_columns:
    display_df[col] = (display_df[col] * 100).round(2)

display_df["Runtime (s)"] = display_df["Runtime (s)"].round(2)


print("\n" + "=" * 90)
print("BASELINE MODEL COMPARISON - DATASET NO. 1")
print("=" * 90)

display(display_df)

Training: Decision Tree
Train Accuracy : 1.0000
Test Accuracy  : 0.9316
Train-Test Gap : 0.0684
Macro Precision: 0.9240
Macro Recall   : 0.9210
Macro F1       : 0.9223
Macro ROC-AUC  : 0.9491
Runtime        : 0.05 s

Training: Random Forest
Train Accuracy : 1.0000
Test Accuracy  : 0.8893
Train-Test Gap : 0.1107
Macro Precision: 0.8781
Macro Recall   : 0.8632
Macro F1       : 0.8693
Macro ROC-AUC  : 0.9809
Runtime        : 0.62 s

Training: Bagging
Train Accuracy : 1.0000
Test Accuracy  : 0.9465
Train-Test Gap : 0.0535
Macro Precision: 0.9396
Macro Recall   : 0.9381
Macro F1       : 0.9386
Macro ROC-AUC  : 0.9945
Runtime        : 18.54 s

Training: XGBoost
Train Accuracy : 0.9994
Test Accuracy  : 0.9527
Train-Test Gap : 0.0466
Macro Precision: 0.9489
Macro Recall   : 0.9436
Macro F1       : 0.9458
Macro ROC-AUC  : 0.9960
Runtime        : 1.21 s


BASELINE MODEL COMPARISON - DATASET NO. 1


,Model,Train Accuracy,Test Accuracy,Gap,Macro Precision,Macro Recall,Macro F1,Macro ROC-AUC,Runtime (s)
0,XGBoost,99.94,95.27,4.66,94.89,94.36,94.58,99.60,1.21
1,Bagging,100.00,94.65,5.35,93.96,93.81,93.86,99.45,18.54
2,Decision Tree,100.00,93.16,6.84,92.40,92.10,92.23,94.91,0.05
3,Random Forest,100.00,88.93,11.07,87.81,86.32,86.93,98.09,0.62


In [5]:
# ============================================================
# STEP 4: CLASS-SPECIFIC BASELINE PERFORMANCE
# ============================================================

from sklearn.metrics import classification_report, confusion_matrix

class_names = list(label_encoder.classes_)

for name, model in fitted_baseline_models.items():

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    y_pred = model.predict(X_test)

    print("\nClassification Report:")

    print(
        classification_report(
            y_test,
            y_pred,
            target_names=class_names,
            digits=4,
            zero_division=0
        )
    )

    print("Confusion Matrix:")
    cm = confusion_matrix(y_test, y_pred)

    cm_df = pd.DataFrame(
        cm,
        index=[f"True {c}" for c in class_names],
        columns=[f"Pred {c}" for c in class_names]
    )

    display(cm_df)


Decision Tree

Classification Report:
              precision    recall  f1-score   support

        Good     0.8699    0.8819    0.8759       144
        Poor     0.9236    0.8808    0.9017       151
   Resistant     0.9582    0.9618    0.9600       262
      Severe     0.9442    0.9595    0.9518       247

    accuracy                         0.9316       804
   macro avg     0.9240    0.9210    0.9223       804
weighted avg     0.9316    0.9316    0.9315       804

Confusion Matrix:


,Pred Good,Pred Poor,Pred Resistant,Pred Severe
True Good,127,3,8,6
True Poor,7,133,3,8
True Resistant,9,1,252,0
True Severe,3,7,0,237



Random Forest

Classification Report:
              precision    recall  f1-score   support

        Good     0.8409    0.7708    0.8043       144
        Poor     0.8456    0.7616    0.8014       151
   Resistant     0.9014    0.9771    0.9377       262
      Severe     0.9246    0.9433    0.9339       247

    accuracy                         0.8893       804
   macro avg     0.8781    0.8632    0.8693       804
weighted avg     0.8872    0.8893    0.8870       804

Confusion Matrix:


,Pred Good,Pred Poor,Pred Resistant,Pred Severe
True Good,111,8,23,2
True Poor,14,115,5,17
True Resistant,5,1,256,0
True Severe,2,12,0,233



Bagging

Classification Report:
              precision    recall  f1-score   support

        Good     0.9054    0.9306    0.9178       144
        Poor     0.9236    0.8808    0.9017       151
   Resistant     0.9770    0.9733    0.9751       262
      Severe     0.9522    0.9676    0.9598       247

    accuracy                         0.9465       804
   macro avg     0.9396    0.9381    0.9386       804
weighted avg     0.9465    0.9465    0.9464       804

Confusion Matrix:


,Pred Good,Pred Poor,Pred Resistant,Pred Severe
True Good,134,3,5,2
True Poor,7,133,1,10
True Resistant,6,1,255,0
True Severe,1,7,0,239



XGBoost

Classification Report:
              precision    recall  f1-score   support

        Good     0.9054    0.9306    0.9178       144
        Poor     0.9640    0.8874    0.9241       151
   Resistant     0.9662    0.9809    0.9735       262
      Severe     0.9602    0.9757    0.9679       247

    accuracy                         0.9527       804
   macro avg     0.9489    0.9436    0.9458       804
weighted avg     0.9530    0.9527    0.9525       804

Confusion Matrix:


,Pred Good,Pred Poor,Pred Resistant,Pred Severe
True Good,134,1,8,1
True Poor,7,134,1,9
True Resistant,5,0,257,0
True Severe,2,4,0,241


In [6]:
# ============================================================
# STEP 5A: XGBOOST OPTIMIZATION
# Dataset No. 1
# ============================================================

import time
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

from xgboost import XGBClassifier


# ------------------------------------------------------------
# 1. Stratified cross-validation
# ------------------------------------------------------------

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)


# ------------------------------------------------------------
# 2. Pipeline
# Feature selection occurs INSIDE cross-validation
# ------------------------------------------------------------

xgb_pipeline = Pipeline([
    
    ("select", SelectKBest(
        score_func=f_classif
    )),
    
    ("model", XGBClassifier(
        objective="multi:softprob",
        num_class=4,
        eval_metric="mlogloss",
        random_state=RANDOM_STATE,
        n_jobs=1,
        tree_method="hist"
    ))
])


# ------------------------------------------------------------
# 3. Hyperparameter search space
# ------------------------------------------------------------

xgb_param_grid = {

    # Feature-selection experiment
    "select__k": [10, 15, 20, 25, "all"],

    # Number of boosting trees
    "model__n_estimators": [100, 200, 400],

    # Tree complexity
    "model__max_depth": [3, 5, 7],

    # Shrinkage
    "model__learning_rate": [0.03, 0.05, 0.1],

    # Row sampling
    "model__subsample": [0.8, 1.0],

    # Feature sampling
    "model__colsample_bytree": [0.8, 1.0]
}


# ------------------------------------------------------------
# 4. Grid search
# ------------------------------------------------------------

xgb_grid = GridSearchCV(
    estimator=xgb_pipeline,
    param_grid=xgb_param_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)


print("Starting XGBoost optimization...")

start = time.time()

xgb_grid.fit(X_train, y_train)

runtime = time.time() - start


# ------------------------------------------------------------
# 5. Best CV configuration
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("BEST XGBOOST CONFIGURATION")
print("=" * 70)

print("\nBest parameters:")

for parameter, value in xgb_grid.best_params_.items():
    print(f"{parameter}: {value}")

print(
    f"\nBest 5-fold CV Macro F1: "
    f"{xgb_grid.best_score_:.4f}"
)

print(
    f"Optimization runtime: "
    f"{runtime / 60:.2f} minutes"
)


# ------------------------------------------------------------
# 6. Final evaluation on LOCKED test set
# ------------------------------------------------------------

best_xgb = xgb_grid.best_estimator_

y_train_pred = best_xgb.predict(X_train)
y_test_pred = best_xgb.predict(X_test)

y_test_proba = best_xgb.predict_proba(X_test)


train_acc = accuracy_score(
    y_train,
    y_train_pred
)

test_acc = accuracy_score(
    y_test,
    y_test_pred
)

gap = train_acc - test_acc

macro_precision = precision_score(
    y_test,
    y_test_pred,
    average="macro",
    zero_division=0
)

macro_recall = recall_score(
    y_test,
    y_test_pred,
    average="macro",
    zero_division=0
)

macro_f1 = f1_score(
    y_test,
    y_test_pred,
    average="macro",
    zero_division=0
)

macro_auc = roc_auc_score(
    y_test,
    y_test_proba,
    multi_class="ovr",
    average="macro"
)


print("\n" + "=" * 70)
print("FINAL XGBOOST TEST PERFORMANCE")
print("=" * 70)

print(f"Train Accuracy : {train_acc:.4f}")
print(f"Test Accuracy  : {test_acc:.4f}")
print(f"Train-Test Gap : {gap:.4f}")
print(f"Macro Precision: {macro_precision:.4f}")
print(f"Macro Recall   : {macro_recall:.4f}")
print(f"Macro F1       : {macro_f1:.4f}")
print(f"Macro ROC-AUC  : {macro_auc:.4f}")


# ------------------------------------------------------------
# 7. Classification report
# ------------------------------------------------------------

print("\nClassification Report:\n")

print(
    classification_report(
        y_test,
        y_test_pred,
        target_names=label_encoder.classes_,
        digits=4,
        zero_division=0
    )
)


# ------------------------------------------------------------
# 8. Confusion matrix
# ------------------------------------------------------------

cm = confusion_matrix(
    y_test,
    y_test_pred
)

cm_df = pd.DataFrame(
    cm,
    index=[
        f"True {c}"
        for c in label_encoder.classes_
    ],
    columns=[
        f"Pred {c}"
        for c in label_encoder.classes_
    ]
)

display(cm_df)

Starting XGBoost optimization...
Fitting 5 folds for each of 540 candidates, totalling 2700 fits

BEST XGBOOST CONFIGURATION

Best parameters:
model__colsample_bytree: 1.0
model__learning_rate: 0.1
model__max_depth: 7
model__n_estimators: 400
model__subsample: 0.8
select__k: all

Best 5-fold CV Macro F1: 0.9489
Optimization runtime: 11.58 minutes

FINAL XGBOOST TEST PERFORMANCE
Train Accuracy : 1.0000
Test Accuracy  : 0.9552
Train-Test Gap : 0.0448
Macro Precision: 0.9504
Macro Recall   : 0.9466
Macro F1       : 0.9478
Macro ROC-AUC  : 0.9967

Classification Report:

              precision    recall  f1-score   support

        Good     0.9073    0.9514    0.9288       144
        Poor     0.9565    0.8742    0.9135       151
   Resistant     0.9735    0.9809    0.9772       262
      Severe     0.9641    0.9798    0.9719       247

    accuracy                         0.9552       804
   macro avg     0.9504    0.9466    0.9478       804
weighted avg     0.9556    0.9552    0.9549   

,Pred Good,Pred Poor,Pred Resistant,Pred Severe
True Good,137,1,6,0
True Poor,9,132,1,9
True Resistant,4,1,257,0
True Severe,1,4,0,242


In [7]:
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()  # repository root
(ROOT / "results" / "figures").mkdir(parents=True, exist_ok=True)
# ============================================================
# Export test-set predictions for Figure 4.
# Paste as the last cell of each notebook (after the optimized
# XGBoost model has been fitted). It uses the notebook's
# best_xgb, label_encoder, X_test and y_test.
# ============================================================
import pandas as pd

DATASET = 1          # set to 2 in the Dataset No. 2 notebook

proba = best_xgb.predict_proba(X_test)
classes = list(label_encoder.classes_)          # Good, Poor, Resistant, Severe
roc = pd.DataFrame(proba, columns=[f"prob_{c}" for c in classes])
roc.insert(0, "true_class", label_encoder.inverse_transform(y_test))
roc.to_csv(ROOT / "results" / f"roc_dataset{DATASET}.csv", index=False)
print(f"Saved roc_dataset{DATASET}.csv with {len(roc)} test samples")

Saved roc_dataset1.csv with 804 test samples


In [ ]:
# ============================================================
# STEP 5B: RANDOM FOREST OPTIMIZATION
# Dataset No. 1
# ============================================================

import time
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)


# ------------------------------------------------------------
# 1. Pipeline
# ------------------------------------------------------------

rf_pipeline = Pipeline([

    ("select", SelectKBest(
        score_func=f_classif
    )),

    ("model", RandomForestClassifier(
        random_state=RANDOM_STATE,
        n_jobs=1
    ))
])


# ------------------------------------------------------------
# 2. Hyperparameter grid
# ------------------------------------------------------------

rf_param_grid = {

    # Feature-selection experiment
    "select__k": [10, 15, 20, 25, "all"],

    # Number of trees
    "model__n_estimators": [200, 400, 600],

    # Tree depth
    "model__max_depth": [None, 10, 20, 30],

    # Minimum samples required to split
    "model__min_samples_split": [2, 5],

    # Minimum samples in leaf
    "model__min_samples_leaf": [1, 2],

    # Number of predictors considered per split
    "model__max_features": ["sqrt", 0.7, 1.0]
}


# ------------------------------------------------------------
# 3. Grid search
# ------------------------------------------------------------

rf_grid = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=rf_param_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)


print("Starting Random Forest optimization...")

start = time.time()

rf_grid.fit(X_train, y_train)

runtime = time.time() - start


# ------------------------------------------------------------
# 4. Best CV configuration
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("BEST RANDOM FOREST CONFIGURATION")
print("=" * 70)

print("\nBest parameters:")

for parameter, value in rf_grid.best_params_.items():
    print(f"{parameter}: {value}")

print(
    f"\nBest 5-fold CV Macro F1: "
    f"{rf_grid.best_score_:.4f}"
)

print(
    f"Optimization runtime: "
    f"{runtime / 60:.2f} minutes"
)


# ------------------------------------------------------------
# 5. Evaluate locked test set
# ------------------------------------------------------------

best_rf = rf_grid.best_estimator_

y_train_pred = best_rf.predict(X_train)
y_test_pred = best_rf.predict(X_test)

y_test_proba = best_rf.predict_proba(X_test)


train_acc = accuracy_score(
    y_train,
    y_train_pred
)

test_acc = accuracy_score(
    y_test,
    y_test_pred
)

gap = train_acc - test_acc

macro_precision = precision_score(
    y_test,
    y_test_pred,
    average="macro",
    zero_division=0
)

macro_recall = recall_score(
    y_test,
    y_test_pred,
    average="macro",
    zero_division=0
)

macro_f1 = f1_score(
    y_test,
    y_test_pred,
    average="macro",
    zero_division=0
)

macro_auc = roc_auc_score(
    y_test,
    y_test_proba,
    multi_class="ovr",
    average="macro"
)


print("\n" + "=" * 70)
print("FINAL RANDOM FOREST TEST PERFORMANCE")
print("=" * 70)

print(f"Train Accuracy : {train_acc:.4f}")
print(f"Test Accuracy  : {test_acc:.4f}")
print(f"Train-Test Gap : {gap:.4f}")
print(f"Macro Precision: {macro_precision:.4f}")
print(f"Macro Recall   : {macro_recall:.4f}")
print(f"Macro F1       : {macro_f1:.4f}")
print(f"Macro ROC-AUC  : {macro_auc:.4f}")


# ------------------------------------------------------------
# 6. Classification report
# ------------------------------------------------------------

print("\nClassification Report:\n")

print(
    classification_report(
        y_test,
        y_test_pred,
        target_names=label_encoder.classes_,
        digits=4,
        zero_division=0
    )
)


# ------------------------------------------------------------
# 7. Confusion matrix
# ------------------------------------------------------------

cm = confusion_matrix(
    y_test,
    y_test_pred
)

cm_df = pd.DataFrame(
    cm,
    index=[
        f"True {c}"
        for c in label_encoder.classes_
    ],
    columns=[
        f"Pred {c}"
        for c in label_encoder.classes_
    ]
)

display(cm_df)

Starting Random Forest optimization...
Fitting 5 folds for each of 720 candidates, totalling 3600 fits


In [ ]:
# ============================================================
# STEP 5C: BAGGING OPTIMIZATION
# Dataset No. 1
# ============================================================

import time
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)


# ------------------------------------------------------------
# 1. Base Decision Tree
# ------------------------------------------------------------

base_tree = DecisionTreeClassifier(
    random_state=RANDOM_STATE
)


# ------------------------------------------------------------
# 2. Pipeline
# ------------------------------------------------------------

bag_pipeline = Pipeline([

    ("select", SelectKBest(
        score_func=f_classif
    )),

    ("model", BaggingClassifier(
        estimator=base_tree,
        random_state=RANDOM_STATE,
        n_jobs=1
    ))
])


# ------------------------------------------------------------
# 3. Hyperparameter grid
# ------------------------------------------------------------

bag_param_grid = {

    # Feature-selection experiment
    "select__k": [10, 15, 20, 25, "all"],

    # Number of base trees
    "model__n_estimators": [50, 100, 200],

    # Fraction of training samples for each tree
    "model__max_samples": [0.7, 0.85, 1.0],

    # Fraction of features available to each tree
    "model__max_features": [0.7, 0.85, 1.0],

    # Tree complexity
    "model__estimator__max_depth": [None, 10, 20],

    # Leaf regularization
    "model__estimator__min_samples_leaf": [1, 2]
}


# ------------------------------------------------------------
# 4. Grid search
# ------------------------------------------------------------

bag_grid = GridSearchCV(
    estimator=bag_pipeline,
    param_grid=bag_param_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)


print("Starting Bagging optimization...")

start = time.time()

bag_grid.fit(X_train, y_train)

runtime = time.time() - start


# ------------------------------------------------------------
# 5. Best CV configuration
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("BEST BAGGING CONFIGURATION")
print("=" * 70)

print("\nBest parameters:")

for parameter, value in bag_grid.best_params_.items():
    print(f"{parameter}: {value}")

print(
    f"\nBest 5-fold CV Macro F1: "
    f"{bag_grid.best_score_:.4f}"
)

print(
    f"Optimization runtime: "
    f"{runtime / 60:.2f} minutes"
)


# ------------------------------------------------------------
# 6. Locked test evaluation
# ------------------------------------------------------------

best_bag = bag_grid.best_estimator_

y_train_pred = best_bag.predict(X_train)
y_test_pred = best_bag.predict(X_test)

y_test_proba = best_bag.predict_proba(X_test)

train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)

gap = train_acc - test_acc

macro_precision = precision_score(
    y_test,
    y_test_pred,
    average="macro",
    zero_division=0
)

macro_recall = recall_score(
    y_test,
    y_test_pred,
    average="macro",
    zero_division=0
)

macro_f1 = f1_score(
    y_test,
    y_test_pred,
    average="macro",
    zero_division=0
)

macro_auc = roc_auc_score(
    y_test,
    y_test_proba,
    multi_class="ovr",
    average="macro"
)


print("\n" + "=" * 70)
print("FINAL BAGGING TEST PERFORMANCE")
print("=" * 70)

print(f"Train Accuracy : {train_acc:.4f}")
print(f"Test Accuracy  : {test_acc:.4f}")
print(f"Train-Test Gap : {gap:.4f}")
print(f"Macro Precision: {macro_precision:.4f}")
print(f"Macro Recall   : {macro_recall:.4f}")
print(f"Macro F1       : {macro_f1:.4f}")
print(f"Macro ROC-AUC  : {macro_auc:.4f}")


# ------------------------------------------------------------
# 7. Classification report
# ------------------------------------------------------------

print("\nClassification Report:\n")

print(
    classification_report(
        y_test,
        y_test_pred,
        target_names=label_encoder.classes_,
        digits=4,
        zero_division=0
    )
)


# ------------------------------------------------------------
# 8. Confusion matrix
# ------------------------------------------------------------

cm = confusion_matrix(y_test, y_test_pred)

cm_df = pd.DataFrame(
    cm,
    index=[
        f"True {c}"
        for c in label_encoder.classes_
    ],
    columns=[
        f"Pred {c}"
        for c in label_encoder.classes_
    ]
)

display(cm_df)

In [ ]:
# ============================================================
# STEP 5D: DECISION TREE OPTIMIZATION
# Dataset No. 1
# ============================================================

import time
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)


# ------------------------------------------------------------
# 1. Pipeline
# ------------------------------------------------------------

dt_pipeline = Pipeline([

    ("select", SelectKBest(
        score_func=f_classif
    )),

    ("model", DecisionTreeClassifier(
        random_state=RANDOM_STATE
    ))
])


# ------------------------------------------------------------
# 2. Hyperparameter grid
# ------------------------------------------------------------

dt_param_grid = {

    # Feature selection
    "select__k": [10, 15, 20, 25, "all"],

    # Split criterion
    "model__criterion": ["gini", "entropy"],

    # Tree complexity
    "model__max_depth": [None, 5, 10, 15, 20, 30],

    # Minimum samples required to split a node
    "model__min_samples_split": [2, 5, 10],

    # Minimum samples in a terminal leaf
    "model__min_samples_leaf": [1, 2, 4],

    # Number of predictors considered at each split
    "model__max_features": [None, "sqrt", 0.7]
}


# ------------------------------------------------------------
# 3. Grid search
# ------------------------------------------------------------

dt_grid = GridSearchCV(
    estimator=dt_pipeline,
    param_grid=dt_param_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)


print("Starting Decision Tree optimization...")

start = time.time()

dt_grid.fit(X_train, y_train)

runtime = time.time() - start


# ------------------------------------------------------------
# 4. Best configuration
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("BEST DECISION TREE CONFIGURATION")
print("=" * 70)

print("\nBest parameters:")

for parameter, value in dt_grid.best_params_.items():
    print(f"{parameter}: {value}")

print(
    f"\nBest 5-fold CV Macro F1: "
    f"{dt_grid.best_score_:.4f}"
)

print(
    f"Optimization runtime: "
    f"{runtime / 60:.2f} minutes"
)


# ------------------------------------------------------------
# 5. Final evaluation on locked test set
# ------------------------------------------------------------

best_dt = dt_grid.best_estimator_

y_train_pred = best_dt.predict(X_train)
y_test_pred = best_dt.predict(X_test)

y_test_proba = best_dt.predict_proba(X_test)


train_acc = accuracy_score(
    y_train,
    y_train_pred
)

test_acc = accuracy_score(
    y_test,
    y_test_pred
)

gap = train_acc - test_acc

macro_precision = precision_score(
    y_test,
    y_test_pred,
    average="macro",
    zero_division=0
)

macro_recall = recall_score(
    y_test,
    y_test_pred,
    average="macro",
    zero_division=0
)

macro_f1 = f1_score(
    y_test,
    y_test_pred,
    average="macro",
    zero_division=0
)

macro_auc = roc_auc_score(
    y_test,
    y_test_proba,
    multi_class="ovr",
    average="macro"
)


# ------------------------------------------------------------
# 6. Print final performance
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL DECISION TREE TEST PERFORMANCE")
print("=" * 70)

print(f"Train Accuracy : {train_acc:.4f}")
print(f"Test Accuracy  : {test_acc:.4f}")
print(f"Train-Test Gap : {gap:.4f}")
print(f"Macro Precision: {macro_precision:.4f}")
print(f"Macro Recall   : {macro_recall:.4f}")
print(f"Macro F1       : {macro_f1:.4f}")
print(f"Macro ROC-AUC  : {macro_auc:.4f}")


# ------------------------------------------------------------
# 7. Classification report
# ------------------------------------------------------------

print("\nClassification Report:\n")

print(
    classification_report(
        y_test,
        y_test_pred,
        target_names=label_encoder.classes_,
        digits=4,
        zero_division=0
    )
)


# ------------------------------------------------------------
# 8. Confusion matrix
# ------------------------------------------------------------

cm = confusion_matrix(
    y_test,
    y_test_pred
)

cm_df = pd.DataFrame(
    cm,
    index=[
        f"True {c}"
        for c in label_encoder.classes_
    ],
    columns=[
        f"Pred {c}"
        for c in label_encoder.classes_
    ]
)

display(cm_df)

In [ ]:
# ============================================================
# STEP 6: MULTI-SEED ROBUSTNESS TEST
# Dataset No. 1
# ============================================================

import time
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from xgboost import XGBClassifier


# ------------------------------------------------------------
# 1. Random seeds
# ------------------------------------------------------------

SEEDS = [0, 1, 2, 3, 4]


# ------------------------------------------------------------
# 2. Function containing FIXED optimized models
# ------------------------------------------------------------

def get_optimized_models(seed):

    models = {

        "Decision Tree": DecisionTreeClassifier(
            criterion="entropy",
            max_depth=None,
            max_features=None,
            min_samples_leaf=1,
            min_samples_split=2,
            random_state=seed
        ),

        "Random Forest": RandomForestClassifier(
            n_estimators=600,
            max_depth=None,
            max_features=1.0,
            min_samples_leaf=1,
            min_samples_split=2,
            random_state=seed,
            n_jobs=-1
        ),

        "Bagging": BaggingClassifier(
            estimator=DecisionTreeClassifier(
                max_depth=20,
                min_samples_leaf=1,
                random_state=seed
            ),
            n_estimators=100,
            max_samples=1.0,
            max_features=1.0,
            random_state=seed,
            n_jobs=-1
        ),

        "XGBoost": XGBClassifier(
            objective="multi:softprob",
            num_class=4,
            eval_metric="mlogloss",
            n_estimators=400,
            max_depth=7,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=1.0,
            tree_method="hist",
            random_state=seed,
            n_jobs=-1
        )
    }

    return models


# ------------------------------------------------------------
# 3. Run experiment
# ------------------------------------------------------------

robustness_results = []

for seed in SEEDS:

    print("\n" + "=" * 75)
    print(f"RANDOM SEED: {seed}")
    print("=" * 75)

    # New stratified 80:20 split
    X_train_seed, X_test_seed, y_train_seed, y_test_seed = (
        train_test_split(
            X,
            y,
            test_size=0.20,
            random_state=seed,
            stratify=y
        )
    )

    models = get_optimized_models(seed)

    for name, model in models.items():

        start = time.time()

        model.fit(
            X_train_seed,
            y_train_seed
        )

        runtime = time.time() - start

        # Predictions
        train_pred = model.predict(X_train_seed)
        test_pred = model.predict(X_test_seed)

        test_proba = model.predict_proba(X_test_seed)

        # Metrics
        train_acc = accuracy_score(
            y_train_seed,
            train_pred
        )

        test_acc = accuracy_score(
            y_test_seed,
            test_pred
        )

        gap = train_acc - test_acc

        macro_precision = precision_score(
            y_test_seed,
            test_pred,
            average="macro",
            zero_division=0
        )

        macro_recall = recall_score(
            y_test_seed,
            test_pred,
            average="macro",
            zero_division=0
        )

        macro_f1 = f1_score(
            y_test_seed,
            test_pred,
            average="macro",
            zero_division=0
        )

        macro_auc = roc_auc_score(
            y_test_seed,
            test_proba,
            multi_class="ovr",
            average="macro"
        )

        robustness_results.append({

            "Seed": seed,
            "Model": name,

            "Train Accuracy": train_acc,
            "Test Accuracy": test_acc,
            "Gap": gap,

            "Macro Precision": macro_precision,
            "Macro Recall": macro_recall,
            "Macro F1": macro_f1,
            "Macro ROC-AUC": macro_auc,

            "Runtime (s)": runtime
        })

        print(
            f"{name:<15} | "
            f"Train={train_acc:.4f} | "
            f"Test={test_acc:.4f} | "
            f"F1={macro_f1:.4f} | "
            f"AUC={macro_auc:.4f} | "
            f"Gap={gap:.4f}"
        )


# ------------------------------------------------------------
# 4. Full results
# ------------------------------------------------------------

robustness_df = pd.DataFrame(
    robustness_results
)

print("\n" + "=" * 90)
print("RESULTS FOR EVERY RANDOM SEED")
print("=" * 90)

display(robustness_df)

In [ ]:
# ============================================================
# STEP 7: SUMMARIZE MULTI-SEED ROBUSTNESS
# ============================================================

metrics = [
    "Train Accuracy",
    "Test Accuracy",
    "Gap",
    "Macro Precision",
    "Macro Recall",
    "Macro F1",
    "Macro ROC-AUC"
]


summary_rows = []

for model_name in robustness_df["Model"].unique():

    model_data = robustness_df[
        robustness_df["Model"] == model_name
    ]

    row = {
        "Model": model_name
    }

    for metric in metrics:

        mean = model_data[metric].mean()
        std = model_data[metric].std(ddof=1)

        row[f"{metric} Mean"] = mean
        row[f"{metric} Std"] = std

    summary_rows.append(row)


summary_df = pd.DataFrame(summary_rows)

# Sort according to mean Macro F1
summary_df = summary_df.sort_values(
    "Macro F1 Mean",
    ascending=False
).reset_index(drop=True)


# ------------------------------------------------------------
# Publication-friendly table
# ------------------------------------------------------------

publication_rows = []

for _, row in summary_df.iterrows():

    publication_rows.append({

        "Model": row["Model"],

        "Train Accuracy (%)":
            f"{row['Train Accuracy Mean']*100:.2f} ± "
            f"{row['Train Accuracy Std']*100:.2f}",

        "Test Accuracy (%)":
            f"{row['Test Accuracy Mean']*100:.2f} ± "
            f"{row['Test Accuracy Std']*100:.2f}",

        "Generalization Gap (pp)":
            f"{row['Gap Mean']*100:.2f} ± "
            f"{row['Gap Std']*100:.2f}",

        "Macro Precision (%)":
            f"{row['Macro Precision Mean']*100:.2f} ± "
            f"{row['Macro Precision Std']*100:.2f}",

        "Macro Recall (%)":
            f"{row['Macro Recall Mean']*100:.2f} ± "
            f"{row['Macro Recall Std']*100:.2f}",

        "Macro F1 (%)":
            f"{row['Macro F1 Mean']*100:.2f} ± "
            f"{row['Macro F1 Std']*100:.2f}",

        "Macro ROC-AUC (%)":
            f"{row['Macro ROC-AUC Mean']*100:.2f} ± "
            f"{row['Macro ROC-AUC Std']*100:.2f}"
    })


publication_df = pd.DataFrame(
    publication_rows
)


print("\n" + "=" * 90)
print("FIVE-SEED ROBUSTNESS SUMMARY - DATASET NO. 1")
print("=" * 90)

display(publication_df)

In [ ]:
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()  # repository root
(ROOT / "results" / "figures").mkdir(parents=True, exist_ok=True)
# ============================================================
# FIGURE 3a
# Train-test accuracy comparison
# Dataset No. 1
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

models = ["Decision Tree", "Random Forest", "Bagging", "XGBoost"]

baseline_train = [100.00, 100.00, 100.00, 99.94]
baseline_test  = [93.16, 88.93, 94.65, 95.27]

optimized_train = [100.00, 100.00, 100.00, 100.00]
optimized_test  = [94.03, 94.53, 94.53, 95.52]

x = np.arange(len(models))
width = 0.19

fig, ax = plt.subplots(figsize=(10, 6))

bars1 = ax.bar(
    x - 1.5*width,
    baseline_train,
    width,
    label="Baseline Train", color='#9ecae1'
)

bars2 = ax.bar(
    x - 0.5*width,
    baseline_test,
    width,
    label="Baseline Test", color='#3182bd'
)

bars3 = ax.bar(
    x + 0.5*width,
    optimized_train,
    width,
    label="Optimized Train", color='#a1d99b'
)

bars4 = ax.bar(
    x + 1.5*width,
    optimized_test,
    width,
    label="Optimized Test", color='#31a354'
)

ax.set_ylabel("Accuracy (%)")
ax.set_xlabel("Machine Learning Model")
ax.set_title("(a) Dataset No. 1")
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.set_ylim(0, 120)
ax.legend(ncol=2)

for bars in [bars1, bars2, bars3, bars4]:
    ax.bar_label(
        bars,
        fmt="%.2f",
        padding=2,
        fontsize=8
    )

plt.tight_layout()

plt.savefig(
    ROOT / "results" / "figures" / "Figure3a_Dataset1_Accuracy.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()  # repository root
(ROOT / "results" / "figures").mkdir(parents=True, exist_ok=True)
# ============================================================
# FIGURE 3a
# Train-test accuracy comparison with robustness error bars
# Dataset No. 1
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

models = [
    "Decision Tree",
    "Random Forest",
    "Bagging",
    "XGBoost"
]

# Baseline results
baseline_train = [100.00, 100.00, 100.00, 99.94]
baseline_test  = [93.16, 88.93, 94.65, 95.27]

# Optimized results
optimized_train = [100.00, 100.00, 100.00, 100.00]
optimized_test  = [94.03, 94.53, 94.53, 95.52]

# Standard deviations from five-seed robustness test
optimized_train_std = [0.00, 0.00, 0.00, 0.00]
optimized_test_std  = [1.08, 0.71, 0.73, 0.64]

x = np.arange(len(models))
width = 0.19

# Larger figure
fig, ax = plt.subplots(figsize=(12, 8))

bars1 = ax.bar(
    x - 1.5 * width,
    baseline_train,
    width,
    label="Baseline Train",
    color="#9ecae1"
)

bars2 = ax.bar(
    x - 0.5 * width,
    baseline_test,
    width,
    label="Baseline Test",
    color="#3182bd"
)

bars3 = ax.bar(
    x + 0.5 * width,
    optimized_train,
    width,
    yerr=optimized_train_std,
    capsize=5,
    label="Optimized Train",
    color="#a1d99b"
)

bars4 = ax.bar(
    x + 1.5 * width,
    optimized_test,
    width,
    yerr=optimized_test_std,
    capsize=5,
    label="Optimized Test",
    color="#31a354"
)

# Larger fonts
ax.set_ylabel(
    "Accuracy (%)",
    fontsize=20,
)

ax.set_xlabel(
    "Machine Learning Model",
    fontsize=20,
)

ax.set_title(
    "(a) Dataset No. 1",
    fontsize=18,
    pad=15
)

ax.set_xticks(x)

ax.set_xticklabels(
    models,
    fontsize=16
)

ax.tick_params(
    axis="y",
    labelsize=16
)

ax.set_ylim(0, 120)

ax.legend(
    ncol=2,
    fontsize=16,
    frameon=True
)

# Numerical values
for bars in [bars1, bars2]:
    ax.bar_label(
        bars,
        fmt="%.2f",
        padding=3,
        fontsize=11
    )

# Extra padding for bars with error bars
for bars in [bars3, bars4]:
    ax.bar_label(
        bars,
        fmt="%.2f",
        padding=8,
        fontsize=12
    )

plt.tight_layout()

plt.savefig(
    ROOT / "results" / "figures" / "Figure3a_Dataset1_Accuracy.png",
    dpi=800,
    bbox_inches="tight"
)

plt.show()

In [ ]:
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()  # repository root
(ROOT / "results" / "figures").mkdir(parents=True, exist_ok=True)
# ============================================================
# FIGURE 4a
# ROC-AUC curves for optimized XGBoost
# Dataset No. 1
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

# Predict probabilities
y_score = best_xgb.predict_proba(X_test)

classes = np.arange(len(label_encoder.classes_))

y_test_bin = label_binarize(
    y_test,
    classes=classes
)

# Larger figure
plt.figure(figsize=(10, 8))

auc_values = []

for i, class_name in enumerate(label_encoder.classes_):

    fpr, tpr, _ = roc_curve(
        y_test_bin[:, i],
        y_score[:, i]
    )

    class_auc = auc(fpr, tpr)
    auc_values.append(class_auc)

    plt.plot(
        fpr,
        tpr,
        linewidth=2.5,
        label=f"{class_name} (AUC = {class_auc:.3f})"
    )

# Random classifier reference
plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    linewidth=2,
    label="Random classifier"
)

macro_auc = np.mean(auc_values)

plt.xlabel(
    "False Positive Rate",
    fontsize=20,
)

plt.ylabel(
    "True Positive Rate",
    fontsize=20,
)

plt.title(
       f"(a) Dataset No. 1\n"
       f"Optimized XGBoost, Macro AUC = {macro_auc:.3f}",
    fontsize=18,
    pad=15
)

plt.xticks(fontsize=16)
plt.yticks(fontsize=16)

plt.legend(
    loc="lower right",
    fontsize=18
)

plt.grid()

plt.xlim([0, 1])
plt.ylim([0, 1.02])

plt.tight_layout()

plt.savefig(
    ROOT / "results" / "figures" / "Figure4a_Dataset1_XGBoost_ROC.png",
    dpi=800,
    bbox_inches="tight"
)

plt.show()

print("Class-specific ROC-AUC:")
for name, score in zip(label_encoder.classes_, auc_values):
    print(f"{name:<12}: {score:.4f}")

print(f"\nMacro ROC-AUC: {macro_auc:.4f}")

In [ ]:
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()  # repository root
(ROOT / "results" / "figures").mkdir(parents=True, exist_ok=True)
# ============================================================
# FIGURE 5a
# XGBoost feature importance
# Dataset No. 1
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt

# Extract XGBoost model from pipeline
xgb_model = best_xgb.named_steps["model"]

feature_names = X.columns

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": xgb_model.feature_importances_
})

# Sort from highest to lowest
importance_df = importance_df.sort_values(
    "Importance",
    ascending=False
).reset_index(drop=True)

print("=" * 70)
print("XGBOOST FEATURE IMPORTANCE - DATASET NO. 1")
print("=" * 70)

display(importance_df)

# Select top 10
top_n = 10

top_features = (
    importance_df
    .head(top_n)
    .sort_values("Importance", ascending=True)
)

# Larger figure
fig, ax = plt.subplots(figsize=(12, 8))

ax.barh(
    top_features["Feature"],
    top_features["Importance"]
)

ax.set_xlabel(
    "Feature Importance",
    fontsize=20,
)

ax.set_ylabel(
    "Feature",
    fontsize=20,
)

ax.set_title(
    "(a) Dataset No. 1",
    fontsize=18,
    pad=15
)

ax.tick_params(
    axis="x",
    labelsize=18
)

ax.tick_params(
    axis="y",
    labelsize=18
)

plt.tight_layout()

plt.savefig(
    ROOT / "results" / "figures" / "Figure5a_Dataset1_FeatureImportance.png",
    dpi=800,
    bbox_inches="tight"
)

plt.show()

In [ ]:
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()  # repository root
(ROOT / "results" / "figures").mkdir(parents=True, exist_ok=True)
# ============================================================
# REDUCED-FEATURE XGBOOST EXPERIMENT: DATASET NO. 1
# (supervisor comment 2)
#
# Standalone script. Reproduces the notebook preprocessing and
# the optimized XGBoost hyperparameters, then retrains XGBoost
# using only the top-k descriptors of its own feature-importance
# ranking and compares with the full descriptor set.
#
# No test-set leakage: the importance ranking is always computed
# from an XGBoost model fitted on the TRAINING subset only.
#
# Dataset No. 1 only. Place this script in the same folder
# as "Dataset No.1.csv" (in the data/ folder of the repository).
#
# Usage:
#   python reduced_feature_xgboost_dataset1.py
#
# Outputs (reduced_feature_results/):
#   dataset{N}_locked_test.csv      random state 42 (as Table 3)
#   dataset{N}_five_seed_raw.csv    seeds 0-4 (as Table 4)
#   dataset{N}_five_seed_summary.csv
#   dataset{N}_top_features.csv     top-k lists used
# ============================================================

import os

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier


# ============================================================
# 1. SETTINGS
# ============================================================

DATASET = 1

CSV_PATH = ROOT / "data" / "Dataset No.1.csv"

# Optimized hyperparameters from the notebooks (grid search,
# Step 5A). Both selected select__k = "all".
XGB_PARAMS = {
    1: dict(n_estimators=400, max_depth=7, learning_rate=0.1,
            subsample=0.8, colsample_bytree=1.0),
}

# Values reported in the manuscript, used as a sanity check
# for the full-descriptor ("all") runs.
EXPECTED = {
    1: {"locked_test_acc": 95.52, "five_seed_test_acc": 95.90},
}

TOP_K_VALUES = [5, 10, 15]     # top-10 corresponds to Figure 5
LOCKED_STATE = 42
SEEDS = [0, 1, 2, 3, 4]
TEST_SIZE = 0.20

OUT_DIR = ROOT / "results" / "reduced_feature_results"
os.makedirs(OUT_DIR, exist_ok=True)


# ============================================================
# 2. LOAD AND PREPROCESS (identical to notebook Steps 1-2)
# ============================================================

df = pd.read_csv(CSV_PATH)
df.columns = df.columns.str.strip()

label_encoder = LabelEncoder()
y = pd.Series(
    label_encoder.fit_transform(df["Behavior"]),
    index=df.index,
    name="Behavior",
)

X = df.drop(columns=["ASTM Material", "Behavior", "Pitting_label"]).copy()

non_numeric = X.select_dtypes(exclude=np.number).columns.tolist()
if non_numeric:
    raise ValueError(f"Non-numeric features found: {non_numeric}")

print("=" * 70)
print(f"REDUCED-FEATURE XGBOOST | DATASET NO. {DATASET}")
print("=" * 70)
print(f"Samples: {len(X)} | Candidate descriptors: {X.shape[1]}")
print("Class encoding:", dict(enumerate(label_encoder.classes_)))

K_LIST = [k for k in TOP_K_VALUES if k < X.shape[1]] + ["all"]


# ============================================================
# 3. HELPERS
# ============================================================

def make_xgb(seed, n_jobs):
    return XGBClassifier(
        objective="multi:softprob",
        num_class=4,
        eval_metric="mlogloss",
        tree_method="hist",
        random_state=seed,
        n_jobs=n_jobs,
        **XGB_PARAMS[DATASET],
    )


def importance_ranking(X_train, y_train, seed, n_jobs):
    """Rank descriptors with XGBoost fitted on training data only."""
    model = make_xgb(seed, n_jobs).fit(X_train, y_train)
    return (
        pd.Series(model.feature_importances_, index=X_train.columns)
        .sort_values(ascending=False)
        .index.tolist()
    )


def evaluate(X_train, X_test, y_train, y_test, seed, n_jobs):
    model = make_xgb(seed, n_jobs).fit(X_train, y_train)
    train_acc = accuracy_score(y_train, model.predict(X_train)) * 100
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)
    test_acc = accuracy_score(y_test, pred) * 100
    return {
        "n_features": X_train.shape[1],
        "Train Accuracy (%)": train_acc,
        "Test Accuracy (%)": test_acc,
        "Generalization Gap (pp)": train_acc - test_acc,
        "Macro F1 (%)": f1_score(y_test, pred, average="macro") * 100,
        "Macro ROC-AUC (%)": roc_auc_score(
            y_test, proba, multi_class="ovr", average="macro") * 100,
    }


# ============================================================
# 4. LOCKED TEST SET (random state 42, as in Table 3)
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=LOCKED_STATE, stratify=y,
)

# n_jobs=1 matches the notebook grid-search model.
ranking = importance_ranking(X_train, y_train, LOCKED_STATE, n_jobs=1)

rows, top_lists = [], []
for k in K_LIST:
    cols = list(X_train.columns) if k == "all" else ranking[:k]
    res = evaluate(X_train[cols], X_test[cols], y_train, y_test,
                   LOCKED_STATE, n_jobs=1)
    rows.append({"k": k, **res})
    if k != "all":
        top_lists.append({"k": k, "features": "; ".join(cols)})

locked = pd.DataFrame(rows).round(2)
locked.to_csv(os.path.join(OUT_DIR, f"dataset{DATASET}_locked_test.csv"),
              index=False)
pd.DataFrame(top_lists).to_csv(
    os.path.join(OUT_DIR, f"dataset{DATASET}_top_features.csv"), index=False)

print("\nLOCKED TEST SET (random state 42)")
print(locked.to_string(index=False))
print("\nTop-10 descriptors (training-set ranking):")
for i, name in enumerate(ranking[:10], start=1):
    print(f"  {i:2d}. {name}")


# ============================================================
# 5. FIVE-SEED ROBUSTNESS (seeds 0-4, as in Table 4)
#    Ranking recomputed on each seed's training subset.
# ============================================================

rows = []
for seed in SEEDS:
    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=seed, stratify=y,
    )
    rank_s = importance_ranking(Xtr, ytr, seed, n_jobs=-1)
    for k in K_LIST:
        cols = list(Xtr.columns) if k == "all" else rank_s[:k]
        res = evaluate(Xtr[cols], Xte[cols], ytr, yte, seed, n_jobs=-1)
        rows.append({"seed": seed, "k": k, **res})

raw = pd.DataFrame(rows)
raw.to_csv(os.path.join(OUT_DIR, f"dataset{DATASET}_five_seed_raw.csv"),
           index=False)

metrics = [c for c in raw.columns if c not in ("seed", "k", "n_features")]
summary = raw.groupby("k", sort=False)[metrics].agg(["mean", "std"]).round(2)
summary.to_csv(os.path.join(OUT_DIR, f"dataset{DATASET}_five_seed_summary.csv"))

print("\nFIVE-SEED SUMMARY (mean, std)")
print(summary.to_string())


# ============================================================
# 6. SANITY CHECK AGAINST THE MANUSCRIPT
# ============================================================

got_locked = float(locked.loc[locked["k"] == "all", "Test Accuracy (%)"].iloc[0])
got_seed = float(
    raw.loc[raw["k"] == "all", "Test Accuracy (%)"].mean().round(2)
)
exp = EXPECTED[DATASET]

print("\nSANITY CHECK (full descriptor set)")
print(f"  Locked test accuracy : {got_locked:.2f}%  "
      f"(manuscript Table 3: {exp['locked_test_acc']:.2f}%)")
print(f"  Five-seed mean       : {got_seed:.2f}%  "
      f"(manuscript Table 4: {exp['five_seed_test_acc']:.2f}%)")

if (abs(got_locked - exp["locked_test_acc"]) > 0.05
        or abs(got_seed - exp["five_seed_test_acc"]) > 0.05):
    print("  WARNING: does not reproduce the manuscript. Check the CSV "
          "file and package versions before using these results.")
else:
    print("  OK: the full-descriptor runs reproduce the manuscript.")